# IA Générative — TP 2 : Types de GANs
**FST — IDS/IGL 5 — 2026-2027**

- **Exercice 1 — DCGAN** (TensorFlow) sur CIFAR-10 (téléchargé automatiquement par TensorFlow).
- **Exercice 2 — StyleGAN3** (PyTorch, dépôt officiel NVIDIA) : génération, interpolation, truncation ψ, fine-tuning.

> **GPU NVIDIA obligatoire pour l'exercice 2** (les scripts StyleGAN3 utilisent CUDA). Sur Google Colab : *Exécution → Modifier le type d'exécution → GPU (T4 ou mieux)*.
> Si la mémoire GPU sature en passant de l'exercice 1 à l'exercice 2, redémarrez la session et exécutez directement la partie StyleGAN3 (elle est indépendante).

# Exercice 1 — DCGAN sur CIFAR-10

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import glob
import time
import numpy as np
import matplotlib.pyplot as plt
import PIL
import imageio.v2 as imageio
import tensorflow as tf
from tensorflow.keras import layers
from IPython import display

# Empêche TensorFlow de réserver toute la mémoire GPU (StyleGAN3 en aura besoin ensuite)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow", tf.__version__, "| GPU :", tf.config.list_physical_devices("GPU"))

## 1. Chargement de CIFAR-10 et normalisation dans [-1, 1]
50 000 images d'entraînement 32×32×3. La normalisation `(x - 127.5) / 127.5` correspond à la sortie `tanh` du générateur.

In [ ]:
(train_images, train_labels), (_, _) = tf.keras.datasets.cifar10.load_data()
train_images = train_images.reshape(train_images.shape[0], 32, 32, 3).astype("float32")
train_images = (train_images - 127.5) / 127.5          # [0, 255] -> [-1, 1]
print(train_images.shape, train_images.min(), train_images.max())

BUFFER_SIZE = 50000
BATCH_SIZE = 256

train_dataset = (tf.data.Dataset.from_tensor_slices(train_images)
                 .shuffle(BUFFER_SIZE)
                 .batch(BATCH_SIZE))
print("Nombre de lots par époque :", len(train_dataset))

In [ ]:
classes = ["avion", "auto", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]
plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(train_images[i] * 0.5 + 0.5)
    plt.title(classes[int(train_labels[i][0])], fontsize=8); plt.axis("off")
plt.show()

## 2. Générateur et discriminateur

In [ ]:
def make_generator_model():
    model = tf.keras.Sequential()
    model.add(layers.Dense(8*8*256, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((8, 8, 256)))
    assert model.output_shape == (None, 8, 8, 256)  # None représente la taille du batch

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(3, (5, 5), strides=(1, 1), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 32, 32, 3)

    return model

### Description détaillée du générateur
Le générateur transforme un vecteur de bruit $z \in \mathbb{R}^{100}$ en une image couleur 32×32×3 en **sur-échantillonnant** progressivement :

| # | Couche | Sortie | Rôle |
|---|---|---|---|
| 1 | `Dense(8*8*256, use_bias=False)` | 16 384 | Projette le bruit dans un espace de grande dimension. Pas de biais car la BatchNorm suivante le rend inutile. |
| 2 | `BatchNormalization` | 16 384 | Normalise les activations → entraînement plus stable, évite l'effondrement des gradients. |
| 3 | `LeakyReLU` | 16 384 | Non-linéarité ; laisse passer un petit gradient pour les valeurs négatives. |
| 4 | `Reshape((8, 8, 256))` | 8×8×256 | Transforme le vecteur en « carte de caractéristiques » spatiale. |
| 5 | `Conv2DTranspose(128, 5×5, stride 2)` + BN + LeakyReLU | 16×16×128 | Convolution transposée : **double la résolution** et réduit le nombre de canaux. |
| 6 | `Conv2DTranspose(64, 5×5, stride 2)` + BN + LeakyReLU | 32×32×64 | Double encore la résolution → taille finale atteinte. |
| 7 | `Conv2DTranspose(3, 5×5, stride 1, tanh)` | 32×32×3 | Produit les 3 canaux RGB ; `tanh` → valeurs dans [-1, 1], comme les images réelles normalisées. |

Principes DCGAN respectés : pas de couches entièrement connectées cachées ni de pooling, BatchNorm dans le générateur, `tanh` en sortie.

In [ ]:
def make_discriminator_model():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[32, 32, 3]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

### Description détaillée du discriminateur
C'est un classifieur CNN binaire (réel / faux) qui **sous-échantillonne** l'image :

| # | Couche | Sortie | Rôle |
|---|---|---|---|
| 1 | `Conv2D(64, 5×5, stride 2)` | 16×16×64 | Extrait des motifs locaux (contours, couleurs) et divise la résolution par 2 (le stride remplace le pooling). |
| 2 | `LeakyReLU` | 16×16×64 | Non-linéarité ; évite les neurones « morts ». |
| 3 | `Dropout(0.3)` | 16×16×64 | Régularisation : empêche le discriminateur de devenir trop fort trop vite. |
| 4 | `Conv2D(128, 5×5, stride 2)` + LeakyReLU + Dropout | 8×8×128 | Caractéristiques de plus haut niveau (formes, textures). |
| 5 | `Flatten` | 8 192 | Vectorise les cartes de caractéristiques. |
| 6 | `Dense(1)` | 1 | **Logit** (pas de sigmoïde) : > 0 → « réel », < 0 → « faux ». La sigmoïde est intégrée dans la perte (`from_logits=True`). |

In [ ]:
generator = make_generator_model()
discriminator = make_discriminator_model()
generator.summary()
discriminator.summary()

## 3. Fonctions de perte et optimiseurs

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)    # vraies images -> 1
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)   # fausses images -> 0
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    # Le générateur veut que ses images soient classées comme réelles (1)
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

## 4. Test du générateur et du discriminateur non entraînés

In [ ]:
noise = tf.random.normal([1, 100])
generated_image = generator(noise, training=False)

plt.imshow((generated_image[0] * 0.5 + 0.5).numpy())
plt.title("Image produite par le générateur NON entraîné"); plt.axis("off")
plt.show()

decision = discriminator(generated_image, training=False)
proba = tf.sigmoid(decision).numpy()[0, 0]
print(f"Logit du discriminateur : {decision.numpy()[0, 0]:.5f}")
print(f"Probabilité 'réelle'    : {proba:.4f} -> classée comme {'RÉELLE' if proba > 0.5 else 'FAUSSE'}")

Sans entraînement, le générateur produit du bruit et le discriminateur répond au hasard (probabilité ≈ 0,5).

## 5. Dimension du bruit et graine fixe de 16 exemples

In [ ]:
EPOCHS = 50
noise_dim = 100
num_examples_to_generate = 16

# Même bruit réutilisé à chaque époque -> on voit l'évolution du générateur sur les mêmes 16 images
seed = tf.random.normal([num_examples_to_generate, noise_dim])

In [ ]:
# (Optionnel) sauvegarde des modèles pendant l'entraînement
checkpoint_dir = "./training_checkpoints"
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

## 6 & 7. `train_step` : génération, pertes, gradients et mise à jour
`@tf.function` compile la fonction en graphe TensorFlow → entraînement beaucoup plus rapide.

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

## 9. `generate_and_save_images`
(définie avant `train`, qui l'appelle)

In [ ]:
IMG_DIR = "dcgan_images"
os.makedirs(IMG_DIR, exist_ok=True)

def generate_and_save_images(model, epoch, test_input):
    # training=False : BatchNorm en mode inférence
    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow((predictions[i] * 0.5 + 0.5).numpy())   # [-1, 1] -> [0, 1]
        plt.axis("off")
    plt.suptitle(f"Époque {epoch}")
    plt.savefig(os.path.join(IMG_DIR, f"image_at_epoch_{epoch:04d}.png"))
    plt.show()

## 8. Fonction `train(dataset, epochs)`

In [ ]:
history = {"gen": [], "disc": []}

def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()
        g_losses, d_losses = [], []

        for image_batch in dataset:
            g, d = train_step(image_batch)
            g_losses.append(float(g)); d_losses.append(float(d))

        history["gen"].append(np.mean(g_losses))
        history["disc"].append(np.mean(d_losses))

        # Images de progression à chaque époque
        display.clear_output(wait=True)
        generate_and_save_images(generator, epoch + 1, seed)

        # Sauvegarde du modèle toutes les 15 époques
        if (epoch + 1) % 15 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

        print(f"Époque {epoch + 1}/{epochs} terminée en {time.time() - start:.1f} s "
              f"| perte G = {history['gen'][-1]:.4f} | perte D = {history['disc'][-1]:.4f}")

    # Images finales
    display.clear_output(wait=True)
    generate_and_save_images(generator, epochs, seed)

In [ ]:
train(train_dataset, EPOCHS)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["gen"], label="Générateur")
plt.plot(history["disc"], label="Discriminateur")
plt.xlabel("Époque"); plt.ylabel("Perte"); plt.legend(); plt.title("Pertes DCGAN")
plt.show()

## 10. `display_image` : afficher l'image d'une époque donnée

In [ ]:
def display_image(epoch_no):
    return PIL.Image.open(os.path.join(IMG_DIR, f"image_at_epoch_{epoch_no:04d}.png"))

display_image(EPOCHS)

In [ ]:
display_image(1)   # comparer avec la première époque

## 11. GIF animé de la progression

In [ ]:
anim_file = "dcgan.gif"

filenames = sorted(glob.glob(os.path.join(IMG_DIR, "image_at_epoch_*.png")))
with imageio.get_writer(anim_file, mode="I", duration=0.2, loop=0) as writer:
    for filename in filenames:
        writer.append_data(imageio.imread(filename))
    writer.append_data(imageio.imread(filenames[-1]))

print(f"GIF créé : {anim_file} ({len(filenames)} images)")
display.Image(filename=anim_file)

**Observation :** CIFAR-10 est beaucoup plus difficile que MNIST (couleurs, 10 classes très variées, arrière-plans). Après ~50 époques, on obtient des formes et des couleurs cohérentes (ciel, herbe, silhouettes d'animaux ou de véhicules) mais rarement des objets nets ; plus d'époques améliorent progressivement le résultat.

# Exercice 2 — StyleGAN3
> Nécessite un **GPU NVIDIA**. Les scripts officiels chargent le modèle sur `cuda`.

## Étape 1 — Explorer avec StyleGAN3

### 1. Cloner le dépôt officiel et installer les dépendances

In [ ]:
import os, sys, glob, subprocess
BASE_DIR = os.getcwd()
SG3_DIR = os.path.join(BASE_DIR, "stylegan3")

if not os.path.isdir(SG3_DIR):
    !git clone https://github.com/NVlabs/stylegan3.git

# ---- Correctifs de compatibilité (dépôt écrit pour PyTorch 1.9)

def patch(rel, pairs):
    p = os.path.join(SG3_DIR, rel)
    src = open(p).read()
    for old, new in pairs:
        if new in src:
            continue                      # déjà patché
        assert old in src, f"motif introuvable dans {rel}: {old[:60]}"
        src = src.replace(old, new)
    open(p, "w").write(src)

# 1) pkg_resources supprimé des setuptools récents
for f in ["torch_utils/ops/conv2d_gradfix.py", "torch_utils/ops/grid_sample_gradfix.py"]:
    patch(f, [("from pkg_resources import parse_version",
               "from packaging.version import parse as parse_version")])

# 2) PyTorch récent : load() renvoie le module mais ne l'ajoute plus à sys.path
patch("torch_utils/custom_ops.py", [
    ("            torch.utils.cpp_extension.load(name=module_name, build_directory=cached_build_dir,",
     "            module = torch.utils.cpp_extension.load(name=module_name, build_directory=cached_build_dir,"),
    ("            torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)",
     "            module = torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)"),
    ("        module = importlib.import_module(module_name)\n",
     "        pass  # module déjà renvoyé par cpp_extension.load()\n"),
])

# 3) Filet de sécurité : si la compilation CUDA échoue, utiliser l'implémentation PyTorch de référence
FALLBACK = '''

#---- patch TP : repli sur l'implémentation de référence si la compilation échoue
_init_cuda = _init
_init_failed = False
def _init():
    global _init_failed
    if _init_failed:
        return False
    try:
        return _init_cuda()
    except Exception as e:
        import warnings
        warnings.warn(f"Plugin CUDA indisponible ({type(e).__name__}: {e}) -> implémentation de référence (plus lente).")
        _init_failed = True
        return False
'''
for f in ["bias_act.py", "upfirdn2d.py", "filtered_lrelu.py"]:
    p = os.path.join(SG3_DIR, "torch_utils/ops", f)
    src = open(p).read()
    if "_init_cuda = _init" not in src:
        marker = "    return True\n"
        i = src.index("def _init():")
        j = src.index(marker, i) + len(marker)
        src = src[:j] + FALLBACK + src[j:]
        open(p, "w").write(src)

print("StyleGAN3 patché ✔")

# PyTorch est déjà installé sur Colab ; en local : pip install torch torchvision (version CUDA)
!pip install -q packaging ninja click requests tqdm imageio imageio-ffmpeg scipy lmdb opencv-python-headless tensorboard

sys.path.insert(0, SG3_DIR)   # pour importer dnnlib / legacy dans le notebook
import torch
print("PyTorch", torch.__version__, "| CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

### 2. Télécharger le modèle pré-entraîné dans `checkpoints/`
Page officielle : <https://catalog.ngc.nvidia.com/orgs/nvidia/teams/research/models/stylegan3>

La cellule ci-dessous télécharge le fichier directement depuis NVIDIA. Vous pouvez aussi le télécharger vous-même depuis la page et le placer dans `checkpoints/`.

In [ ]:
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
NGC_URL = "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/"

def download_ckpt(name):
    path = os.path.join(CKPT_DIR, name)
    if not os.path.exists(path):
        print("Téléchargement de", name, "...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", path, NGC_URL + name], check=True)
    else:
        print(name, "déjà présent")
    return path

FFHQ_PKL = download_ckpt("stylegan3-r-ffhq-1024x1024.pkl")

### 3. Générer des images avec `gen_images.py` (trunc=1, seeds=2,5,8)

In [ ]:
%cd {BASE_DIR}
!python stylegan3/gen_images.py --outdir=out --trunc=1 --seeds=2,5,8 --network=checkpoints/stylegan3-r-ffhq-1024x1024.pkl

### 4. Afficher les images générées

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def show_images(paths, titles=None, size=4):
    plt.figure(figsize=(size * len(paths), size))
    for i, p in enumerate(paths):
        plt.subplot(1, len(paths), i + 1)
        plt.imshow(Image.open(p)); plt.axis("off")
        plt.title(titles[i] if titles else os.path.basename(p))
    plt.show()

show_images([f"out/seed{s:04d}.png" for s in (2, 5, 8)])

### 5. Interpolation entre deux styles → `interpolation.mp4`
On charge le générateur dans le notebook, on calcule les vecteurs de style $w_1$ et $w_2$ de deux seeds (réseau de *mapping*), puis on interpole linéairement $w_t = (1-t)\,w_1 + t\,w_2$ et on synthétise chaque image intermédiaire.

In [ ]:
import dnnlib, legacy
import imageio.v2 as imageio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_G(pkl):
    with dnnlib.util.open_url(pkl) as f:
        return legacy.load_network_pkl(f)["G_ema"].to(device).eval()

@torch.no_grad()
def seed_to_w(G, seed, psi=1.0):
    z = torch.from_numpy(np.random.RandomState(seed).randn(1, G.z_dim)).to(device)
    c = torch.zeros([1, G.c_dim], device=device)
    return G.mapping(z, c, truncation_psi=psi)

@torch.no_grad()
def synth(G, w):
    img = G.synthesis(w, noise_mode="const")
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    return img[0].cpu().numpy()

def interpolation_video(G, seed_a, seed_b, out="interpolation.mp4", n_frames=90, psi=1.0, fps=30, size=512):
    w_a, w_b = seed_to_w(G, seed_a, psi), seed_to_w(G, seed_b, psi)
    frames = []
    with imageio.get_writer(out, fps=fps, codec="libx264", quality=8, macro_block_size=1) as writer:
        for t in np.linspace(0, 1, n_frames):
            frame = synth(G, (1 - t) * w_a + t * w_b)
            frame = np.array(Image.fromarray(frame).resize((size, size), Image.LANCZOS))
            writer.append_data(frame)
            frames.append(frame)
    return frames

G_ffhq = load_G(FFHQ_PKL)
frames = interpolation_video(G_ffhq, 2, 8, out="interpolation.mp4")
print("Vidéo enregistrée : interpolation.mp4")

# Aperçu : 8 étapes de la transition
idx = np.linspace(0, len(frames) - 1, 8).astype(int)
plt.figure(figsize=(24, 3))
for k, i in enumerate(idx):
    plt.subplot(1, 8, k + 1); plt.imshow(frames[i]); plt.axis("off"); plt.title(f"t = {i / (len(frames) - 1):.2f}")
plt.show()

In [ ]:
from IPython.display import Video
Video("interpolation.mp4", embed=True, width=512)

*Alternative avec le script officiel* (interpolation en boucle entre plusieurs seeds) :
```
!python stylegan3/gen_video.py --output=interpolation_script.mp4 --trunc=1 --seeds=2,5,8 --grid=1x1 --network=checkpoints/stylegan3-r-ffhq-1024x1024.pkl
```

### 6. Expérimenter avec *Truncation ψ*
ψ rapproche le vecteur de style de la **moyenne** $\bar{w}$ : $w' = \bar{w} + \psi\,(w - \bar{w})$.
- ψ petit (0,3–0,5) → visages très réalistes mais **peu diversifiés** (proches du « visage moyen »).
- ψ = 1 → pas de troncature, diversité maximale.
- ψ > 1 → plus original mais plus d'artefacts.

In [ ]:
PSIS = [0.3, 0.5, 0.7, 1.0, 1.2]
SEEDS = "2,5,8"

for psi in PSIS:
    !python stylegan3/gen_images.py --outdir=out/trunc_{psi} --trunc={psi} --seeds={SEEDS} --network=checkpoints/stylegan3-r-ffhq-1024x1024.pkl

### 7. Grille de comparaison seeds × truncation

In [ ]:
def show_grid(seeds, psis, root="out"):
    fig, axes = plt.subplots(len(seeds), len(psis), figsize=(3 * len(psis), 3 * len(seeds)))
    for r, s in enumerate(seeds):
        for c, psi in enumerate(psis):
            ax = axes[r, c]
            ax.imshow(Image.open(f"{root}/trunc_{psi}/seed{s:04d}.png").resize((256, 256)))
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0: ax.set_title(f"ψ = {psi}")
            if c == 0: ax.set_ylabel(f"seed {s}", fontsize=12)
    plt.tight_layout(); plt.show()

show_grid([2, 5, 8], PSIS)

Chaque ligne = un seed, chaque colonne = une valeur de ψ. Plus ψ diminue, plus les 3 visages se ressemblent (convergence vers le visage moyen) ; plus il augmente, plus les traits deviennent marqués et variés.

### 8. Autres types d'images en changeant de checkpoint
Checkpoints StyleGAN3 officiels : `afhqv2` (animaux : chats, chiens, animaux sauvages) et `metfaces` (portraits d'œuvres d'art).
NVIDIA ne publie pas de modèle StyleGAN3 de **paysages** : pour cela, téléchargez un checkpoint communautaire (fichier `.pkl` compatible StyleGAN3, ex. entraîné sur LHQ) et mettez son chemin dans `CUSTOM_PKL`.

In [ ]:
AFHQ_PKL = download_ckpt("stylegan3-r-afhqv2-512x512.pkl")
METF_PKL = download_ckpt("stylegan3-t-metfaces-1024x1024.pkl")

!python stylegan3/gen_images.py --outdir=out/afhq --trunc=0.7 --seeds=0-5 --network={AFHQ_PKL}
!python stylegan3/gen_images.py --outdir=out/metfaces --trunc=0.7 --seeds=0-5 --network={METF_PKL}

show_images(sorted(glob.glob("out/afhq/*.png")), size=3)
show_images(sorted(glob.glob("out/metfaces/*.png")), size=3)

In [ ]:
# Checkpoint personnalisé (paysages, objets...) téléchargé par vos soins
CUSTOM_PKL = "checkpoints/mon_modele.pkl"   # <- à modifier

if os.path.exists(CUSTOM_PKL):
    !python stylegan3/gen_images.py --outdir=out/custom --trunc=0.7 --seeds=0-5 --network={CUSTOM_PKL}
    show_images(sorted(glob.glob("out/custom/*.png")), size=3)
else:
    print("Placez un checkpoint dans", CUSTOM_PKL)

In [ ]:
# Libérer la mémoire GPU avant le fine-tuning
del G_ffhq
torch.cuda.empty_cache()

## Étape 2 — Fine-tuning de StyleGAN3
> Le fine-tuning à 1024×1024 demande plusieurs GPU haut de gamme. Sur Colab (T4 / L4 / A100 unique), on travaille en **256×256** en partant du checkpoint `stylegan3-r-ffhqu-256x256.pkl` (la résolution du dataset doit être **identique** à celle du modèle de départ).

### 1. Préparation du dataset
Téléchargez **Danbooru2019 Portraits** (ou tout autre dataset d'images) et placez les images dans le dossier `RAW_DIR` (sous-dossiers autorisés).
Étapes :
1. **Redimensionnement** (recadrage central carré + resize à `RES`×`RES`).
2. **Conversion en LMDB** (base clé → image encodée) pour accélérer la lecture.
3. `dataset_tool.py` lit la base LMDB et produit l'archive `.zip` attendue par `train.py` (StyleGAN3 s'entraîne sur un zip ou un dossier, pas directement sur une LMDB).

In [ ]:
RAW_DIR = os.path.join(BASE_DIR, "danbooru_portraits")   # <- vos images téléchargées
RES = 256                                                 # 256 (Colab) ou 1024 (multi-GPU)
MAX_IMAGES = 5000                                         # limiter pour un TP (None = tout)

RESIZED_DIR = os.path.join(BASE_DIR, f"dataset_{RES}")
LMDB_DIR = os.path.join(BASE_DIR, f"dataset_{RES}_lmdb")  # le nom DOIT finir par _lmdb (dataset_tool.py)
ZIP_PATH = os.path.join(BASE_DIR, "datasets", f"custom-{RES}x{RES}.zip")

exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
raw_files = sorted(p for p in glob.glob(os.path.join(RAW_DIR, "**", "*"), recursive=True) if p.lower().endswith(exts))
if MAX_IMAGES:
    raw_files = raw_files[:MAX_IMAGES]
print(len(raw_files), "images trouvées dans", RAW_DIR)

In [ ]:
from tqdm.auto import tqdm

def center_crop_resize(path, res):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    return img.resize((res, res), Image.LANCZOS)

os.makedirs(RESIZED_DIR, exist_ok=True)
for i, p in enumerate(tqdm(raw_files, desc="Redimensionnement")):
    try:
        center_crop_resize(p, RES).save(os.path.join(RESIZED_DIR, f"{i:07d}.png"))
    except Exception as e:
        print("Ignorée :", p, e)

resized = sorted(glob.glob(os.path.join(RESIZED_DIR, "*.png")))
print(len(resized), "images", Image.open(resized[0]).size)
show_images(resized[:6], size=2.5)

In [ ]:
import io, lmdb

def build_lmdb(image_paths, lmdb_dir):
    map_size = sum(os.path.getsize(p) for p in image_paths) * 2 + (1 << 26)
    env = lmdb.open(lmdb_dir, map_size=map_size)
    with env.begin(write=True) as txn:
        for i, p in enumerate(tqdm(image_paths, desc="LMDB")):
            with open(p, "rb") as f:
                txn.put(f"{i:08d}".encode(), f.read())     # valeur = image PNG encodée
    env.close()

build_lmdb(resized, LMDB_DIR)
with lmdb.open(LMDB_DIR, readonly=True, lock=False) as env:
    print("Entrées dans la LMDB :", env.stat()["entries"])

In [ ]:
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
!python stylegan3/dataset_tool.py --source={LMDB_DIR} --dest={ZIP_PATH} --resolution={RES}x{RES}

### 2. Fine-tuning à partir d'un modèle pré-entraîné (`--resume`)
Principaux paramètres :
- `--cfg=stylegan3-r` : même configuration que le checkpoint de départ.
- `--resume` : poids pré-entraînés (FFHQ 256) → apprentissage par transfert.
- `--gamma` : poids de la régularisation R1 (petit pour 256×256).
- `--mirror=1` : augmentation par symétrie horizontale ; `--aug=ada` (par défaut) : augmentation adaptative, indispensable avec peu d'images.
- `--kimg` : durée totale (milliers d'images vues) ; `--snap` : fréquence des snapshots (en *ticks* de `--tick` kimg).
- `--metrics=none` : désactive le calcul du FID (très long).

In [ ]:
if RES == 256:
    RESUME_PKL = download_ckpt("stylegan3-r-ffhqu-256x256.pkl")
else:
    RESUME_PKL = download_ckpt("stylegan3-r-ffhqu-1024x1024.pkl")

KIMG = 40          # quelques dizaines de kimg suffisent pour voir l'adaptation (~1 h sur T4)
BATCH = 16
BATCH_GPU = 8      # réduire à 4 si "CUDA out of memory"
GAMMA = 2

!python stylegan3/train.py --outdir=training-runs --cfg=stylegan3-r --data={ZIP_PATH} \
    --gpus=1 --batch={BATCH} --batch-gpu={BATCH_GPU} --gamma={GAMMA} --mirror=1 \
    --kimg={KIMG} --tick=1 --snap=5 --metrics=none --resume={RESUME_PKL}

> La première exécution compile les opérateurs CUDA personnalisés (quelques minutes). Si la compilation échoue, StyleGAN3 bascule sur une implémentation de référence plus lente, mais fonctionnelle.

### 3. Suivi avec TensorBoard
`train.py` écrit dans `training-runs/<run>/` : les snapshots `network-snapshot-XXXXXX.pkl`, les grilles d'images `fakesXXXXXX.png`, les statistiques `training_stats.jsonl` et les journaux `*.tfevents` pour TensorBoard.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir training-runs

In [ ]:
run_dir = sorted(glob.glob("training-runs/*"))[-1]
print("Dossier du run :", run_dir)
fakes = sorted(glob.glob(os.path.join(run_dir, "fakes*.png")))
print([os.path.basename(f) for f in fakes])

# Première grille (modèle de départ) vs dernière grille (après fine-tuning)
for f in [fakes[0], fakes[-1]]:
    plt.figure(figsize=(12, 7))
    plt.imshow(Image.open(f)); plt.axis("off"); plt.title(os.path.basename(f))
    plt.show()

### 4. Générer des images avec le modèle fine-tuné

In [ ]:
snapshots = sorted(glob.glob(os.path.join(run_dir, "network-snapshot-*.pkl")))
FT_PKL = snapshots[-1]
print("Snapshot utilisé :", FT_PKL)

!python stylegan3/gen_images.py --outdir=out/finetuned --trunc=0.7 --seeds=0-7 --network={FT_PKL}

show_images(sorted(glob.glob("out/finetuned/*.png"))[:4], size=3)
show_images(sorted(glob.glob("out/finetuned/*.png"))[4:8], size=3)

In [ ]:
# Comparaison même seed : modèle de départ (FFHQ) vs modèle fine-tuné
!python stylegan3/gen_images.py --outdir=out/before_ft --trunc=0.7 --seeds=0-3 --network={RESUME_PKL}

show_images([f"out/before_ft/seed{s:04d}.png" for s in range(4)], [f"Avant — seed {s}" for s in range(4)], size=3)
show_images([f"out/finetuned/seed{s:04d}.png" for s in range(4)], [f"Après — seed {s}" for s in range(4)], size=3)

**Observation :** avec les mêmes seeds, le modèle fine-tuné conserve une partie de la structure du modèle de départ (pose, cadrage) mais adopte le style du nouveau dataset (traits de dessin anime, couleurs, contours). Le transfert depuis un modèle pré-entraîné converge beaucoup plus vite qu'un entraînement à partir de zéro.